# CPIC on Drift-Diffusion Data Generation

In [ ]:
import numpy as np
import torch

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import sys
from pathlib import Path
print(sys.executable)

# Paths: data_generation (for generate_drift_diffusion), and src (for cpic package)
data_generation_path = Path("../data_generation").resolve()
src_path = Path("../src").resolve()
for p in (data_generation_path, src_path):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from generate_drift_diffusion import (
    generate_drift_diffusion_positions,
    generate_drift_diffusion_process_timeseries,
    animate_drift_diffusion_process,
    plot_verification_3d,
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from cpic import CPIC
from cpic.utils.data import PastFutureDataset

# Simulate the Drift-Diffusion Data Generation Process

In [ ]:
num_noise = 50  # modify depending on how much noise you want

In [ ]:
ani = animate_drift_diffusion_process(t_max=200, 
                                      num_blob=30, 
                                      num_noise=num_noise, 
                                      orbit_radius=3.0, 
                                      omega=0.05, 
                                      sigma_blob=0.1, 
                                      sigma_noise=0.5, 
                                      spatial_bounds=10.0,
                                      seed=42, 
                                      interval=50)
plt.close(ani._fig)
HTML(ani.to_jshtml())

In [ ]:
positions = generate_drift_diffusion_positions(t_max=500,
                                               num_blob=30,
                                               num_noise=num_noise,
                                               orbit_radius=3.0,
                                               omega=0.05,
                                               sigma_blob=0.1,
                                               sigma_noise=0.5,
                                               spatial_bounds=10.0,
                                               seed=42)
fig, ax = plot_verification_3d(positions, num_blob=30, num_noise=num_noise)
plt.show()

# Generate the Data (interleaved particle timeseries)

Observations are shape `(t_max, N*2)`: particles sorted by **x at t=0**, columns `[x0,y0,x1,y1,...]`, z-scored per column. CPIC uses this as `xdim = N*2`.


In [ ]:
data, gt_latent, particle_order, particle_labels = generate_drift_diffusion_process_timeseries(
    t_max=500,
    num_blob=30,
    num_noise=num_noise,
    orbit_radius=3.0,
    omega=0.05,
    sigma_blob=0.1,
    sigma_noise=0.5,
    spatial_bounds=10.0,
    seed=42,
)
n_spatial = data.shape[1]
print("data shape", data.shape)
print("particle_order (x-sort at t=0):", particle_order.shape, "labels (blob=1/noise=0, orig order):", particle_labels.shape)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(data.T, aspect="auto")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Feature index (interleaved x,y; x-sorted)")
axes[0].set_title("Features x time\n(particle timeseries)")

axes[1].imshow(data, aspect="auto")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("Time")
axes[1].set_title("Time x features")

axes[2].plot(gt_latent[:, 0], gt_latent[:, 1], "b--", alpha=0.7)
axes[2].set_xlabel("cos($\omega$ t)")
axes[2].set_ylabel("sin($\omega$ t)")
axes[2].set_aspect("equal")
axes[2].set_title("GT circular latent motion")

plt.tight_layout()
plt.show()


In [ ]:
def animate_particle_timeseries(data, interval=80):
    T, F = data.shape
    fig, ax = plt.subplots(figsize=(12, 2.5))
    lo, hi = float(data.min()), float(data.max())
    im = ax.imshow(data[0:1], aspect="auto", vmin=lo, vmax=hi, cmap="viridis")
    ax.set_xlabel("Feature index (x0,y0,x1,y1,... after x-sort at t=0)")
    ax.set_ylabel("frame")
    plt.colorbar(im, ax=ax)
    def update(frame):
        im.set_data(data[frame : frame + 1])
        ax.set_title(f"t = {frame}")
        return (im,)
    return animation.FuncAnimation(fig, update, frames=T, interval=interval, blit=False)

anim1 = animate_particle_timeseries(data)
plt.close(anim1._fig)
HTML(anim1.to_jshtml())


# Run CPIC

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "mps"
print("device:", device)

In [ ]:
T = 10
train_data = PastFutureDataset([data], window_size=T)

### Helper functions 

In [ ]:
def build_past_windows_and_gt(data, gt_latent, window_size, model):
    """Build past windows, encode them, and align ground truth to window end-times."""
    # data: (n_timesteps, n_spatial)
    # window_size (T): number of timesteps per past window
    # past_windows: (num_windows, T, n_spatial) where num_windows = n_timesteps - T
    past_windows = np.stack([data[t - window_size : t] for t in range(window_size, len(data))], axis=0)
    past_tensor = torch.from_numpy(past_windows).float().to(device)
    with torch.no_grad():
        encoded_windows = model.encode(past_tensor) # (num_windows, T, ydim=2)
        # Use the last latent in each window: corresponds to time index t-1
        z = encoded_windows[:, -1, :].cpu().numpy() # (num_windows, ydim=2)

    # Align ground truth mu(t): for window ending at time (t-1), pick gt_latent[t-1]
    gt = gt_latent[np.arange(window_size, len(data)) - 1]

    return z, gt

def evaluate_cpic_linear_decode(z, centroid_true, encoder_name="mlp"):
    """Fit linear probe z to true mu(t)"""
    reg = LinearRegression().fit(z, centroid_true)
    centroid_pred = reg.predict(z)
    r2_x = r2_score(centroid_true[:, 0], centroid_pred[:, 0])
    r2_y = r2_score(centroid_true[:, 1], centroid_pred[:, 1])

    print(f"{encoder_name} — CPIC linear decode (R^2_x={r2_x:.2f}, R^2_y={r2_y:.2f})")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(centroid_true[:, 0], centroid_true[:, 1], "b--", label="Ground truth $\mu(t)$", linewidth=2)
    ax.plot(centroid_pred[:, 0], centroid_pred[:, 1], "r-",
            label=f"Linear decode ($R^2_x$={r2_x:.2f}, $R^2_y$={r2_y:.2f})")
    ax.set_title(f"CPIC latent vs ground truth centroid\n(encoder: {encoder_name})")
    ax.axis("equal")
    ax.legend(loc="upper right")
    fig.tight_layout(rect=[0, 0, 0.72, 1])
    plt.show()

In [ ]:
import gc

def cleanup_torch(verbose=True):
    """Release Python refs and ask torch backends to free cached memory."""
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        if verbose:
            alloc = torch.cuda.memory_allocated() / (1024 ** 2)
            reserved = torch.cuda.memory_reserved() / (1024 ** 2)
            print(f"[cleanup] CUDA allocated={alloc:.1f} MB, reserved={reserved:.1f} MB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        if verbose:
            print("[cleanup] MPS backend active (no torch empty_cache equivalent).")
    elif verbose:
        print("[cleanup] CPU backend active.")


def cleanup_model_resources(*objs, verbose=True):
    """Drop references to large training objects, then clear backend caches."""
    for obj in objs:
        del obj
    cleanup_torch(verbose=verbose)

## MLP encoder

In [ ]:
encoder_params = {
    "deterministic": False,
    "encoder_type": "mlp",
    "linear_encoder": False,
    "n_layers": 1,
    "activation": "relu",
}

In [ ]:
cpic = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params, hidden_dim=64, beta=1e-5, device=device)
cpic.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic.fit(X=train_data, epochs=50, batch_size=128, lr=3e-4, early_stop=20)

In [ ]:
z, gt = build_past_windows_and_gt(data, gt_latent, T, cpic)
evaluate_cpic_linear_decode(z, gt, encoder_name="MLP")

In [ ]:
cleanup_model_resources(cpic, loss, I_compress, I_predictive)

## ConvSpatial (Latent predictive space) encoder

`conv_spatial` applies Conv2d along the **feature** axis (particle channels), suitable for the `(t_max, N*2)` interleaved representation.


In [ ]:
encoder_params = {
    "deterministic": False,
    "encoder_type": "conv_spatial",
    "linear_encoder": False,
    "n_layers": 1,
    "activation": "relu",
    "conv_kernel_size": 4,
    "conv_stride": 2,
    "conv_padding": 1,
}


In [ ]:
cpic_conv_latent = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params, hidden_dim=64, beta=1e-5, device=device)
cpic_conv_latent.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic_conv_latent.fit(X=train_data, epochs=50, batch_size=128, lr=3e-4, early_stop=20)


In [ ]:
z, gt = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_latent)
evaluate_cpic_linear_decode(z, gt, encoder_name="ConvSpatial (L)")

### Kernel / feature analysis

The kernels slide along the **feature** axis (interleaved particle coordinates). We aggregate importance per input feature, then per particle (sum of |w| over x and y channels), scattered at **t=0** positions in **x-sorted order** (same as the timeseries columns).


In [ ]:
enc = cpic_conv_latent.encoder
layer_idx = 0

if getattr(enc, "encoder_type", None) != "conv_spatial":
    print("Skipping kernel analysis: encoder is not conv_spatial")
else:
    w, meta = enc.get_filters(layer_idx=layer_idx)
    K_h, K_w = meta["kernel_size"]
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "padding:", meta["padding"], "stride:", meta["stride"])

In [ ]:
def _pad_filter_magnitudes(w_mag, gap_rows=1):
    """
    Helper function to pad the filter magnitudes with NaNs between filters for easier visualization.
    """
    C_out, Kh = w_mag.shape
    pad_h = C_out + (C_out - 1) * gap_rows
    pad = np.full((pad_h, Kh), np.nan, dtype=np.float64)
    for i in range(C_out):
        pad[i * (1 + gap_rows), :] = w_mag[i, :]
    return pad, C_out, Kh, gap_rows

def plot_filter_heatmap_panels(w_mag, gap_rows=1, filters_per_panel=12, suptitle=None, cell_inches=0.28):
    """
    Plot each filter magnitude as a heatmap.
    """
    C_out, Kh = w_mag.shape

    chunks = []
    for start in range(0, C_out, filters_per_panel):
        end = min(start + filters_per_panel, C_out)
        sub = w_mag[start:end, :]
        pad, c_sub, kh, g = _pad_filter_magnitudes(sub, gap_rows=gap_rows)
        chunks.append((pad, start, end, c_sub, kh, g))
    n_panels = len(chunks)
    max_pad_h = max(pad.shape[0] for pad, *_ in chunks)

    # specific fig size for this plot for better visualization
    fig_w = n_panels * (Kh * cell_inches) + max(0, n_panels - 1)
    fig_h = max_pad_h * cell_inches
    fig, axes = plt.subplots(1, n_panels, figsize=(fig_w, fig_h))

    ims = []
    for ax, (pad, start, end, c_sub, kh, g) in zip(axes.flat, chunks):
        im = ax.imshow(pad, aspect="equal", cmap="magma")
        ims.append(im)

        ax.set_xticks(np.arange(kh))
        ax.set_xticklabels([str(i) for i in range(kh)])

        ax.set_yticks([i * (1 + g) for i in range(c_sub)])
        ax.set_yticklabels([str(start + i) for i in range(c_sub)])

        ax.set_title("filters {}-{}".format(start, end - 1), fontsize=10)
    axes.flat[0].set_ylabel("Output channel")

    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
        
    cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), shrink=0.7, pad=0.02)
    cbar.set_label("|weight|")

    return fig, axes

w_mag = np.abs(w[:, 0, :, 0]) # (C_out, K_h)
fig, axes = plot_filter_heatmap_panels(w_mag, gap_rows=1, filters_per_panel=8, suptitle=f"conv_spatial mean layer {layer_idx}")
plt.show()

In [ ]:
def conv_input_feature_importance(w, input_dim, padding, stride):
    """
    Helper function to compute the importance of each input feature in the conv_spatial layer.
    """
    # w: (C_out, 1, K, 1) for first-layer conv
    _, _, K, _ = w.shape
    p, _ = padding
    s, _ = stride
    imp = np.zeros(input_dim, dtype=np.float64)
    w_mag = np.abs(w[:, 0, :, 0]) # (C, K)
    L_out = (input_dim + 2 * p - K) // s + 1
    for o in range(L_out):
        for k in range(K):
            j = o * s - p + k # j is the index of the input feature that contributes to the output channel o
            if 0 <= j < input_dim:
                imp[j] += float(w_mag[:, k].sum())
    return imp

imp_feat = conv_input_feature_importance(w, n_spatial, meta["padding"], meta["stride"])
assert 2 * len(particle_order) == n_spatial, (2 * len(particle_order), n_spatial)
imp_particle = imp_feat[0::2] + imp_feat[1::2] # sum x and y channel importance per particle (sorted order); interleaved x,y

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(positions[0, particle_order, 0], positions[0, particle_order, 1], c=imp_particle, edgecolors="k", linewidths=0.3)
ax.set_aspect("equal")
ax.set_xlabel("x (t=0)")
ax.set_ylabel("y (t=0)")
ax.set_title(f"Per-particle feature importance (summed |weights|, layer {layer_idx})")
plt.colorbar(sc, ax=ax, label="importance")
plt.tight_layout()
plt.show()

In [ ]:
cleanup_model_resources(cpic_conv_latent, loss, I_compress, I_predictive)

## ConvSpatial (Observation predictive space) encoder

In [ ]:
cpic_conv_obs = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params, hidden_dim=64, beta=1e-5, device=device, predictive_space="observation")
cpic_conv_obs.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic_conv_obs.fit(X=train_data, epochs=50, batch_size=128, lr=3e-4, early_stop=20)


In [ ]:
z, gt = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_obs)
evaluate_cpic_linear_decode(z, gt, encoder_name="ConvSpatial (O)")

### Kernel / feature analysis

In [ ]:
enc_obs = cpic_conv_obs.encoder
layer_idx = 0

if getattr(enc_obs, "encoder_type", None) != "conv_spatial":
    print("Skipping kernel analysis: encoder is not conv_spatial")
else:
    w, meta = enc_obs.get_filters(layer_idx=layer_idx)
    K_h, K_w = meta["kernel_size"]
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "padding:", meta["padding"], "stride:", meta["stride"])

In [ ]:
w_mag = np.abs(w[:, 0, :, 0]) # (C_out, K_h)
fig, axes = plot_filter_heatmap_panels(w_mag, gap_rows=1, filters_per_panel=8, suptitle=f"conv_spatial mean layer {layer_idx}")
plt.show()

In [ ]:
imp_feat = conv_input_feature_importance(w, n_spatial, meta["padding"], meta["stride"])
assert 2 * len(particle_order) == n_spatial, (2 * len(particle_order), n_spatial)
imp_particle = imp_feat[0::2] + imp_feat[1::2] # sum x and y channel importance per particle (sorted order); interleaved x,y

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(positions[0, particle_order, 0], positions[0, particle_order, 1], c=imp_particle, edgecolors="k", linewidths=0.3)
ax.set_aspect("equal")
ax.set_xlabel("x (t=0)")
ax.set_ylabel("y (t=0)")
ax.set_title(f"Per-particle feature importance (summed |weights|, layer {layer_idx})")
plt.colorbar(sc, ax=ax, label="importance")
plt.tight_layout()
plt.show()

In [ ]:
cleanup_model_resources(cpic_conv_obs, loss, I_compress, I_predictive)